# System Recommenders - Final Project 2025

### 🎯 Objective

Develop a recommender system that suggests short videos to users based on user preferences, interaction histories, and video content using the KuaiRec dataset. The challenge is to create a personalised and scalable recommendation engine similar to those used in platforms like TikTok or Kuaishou.

### 📥 Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

plt.rcParams["figure.figsize"] = (20, 13)
%matplotlib inline
%config InlineBackend.figure_format = "retina"

### 📊 Download Dataset

We will use the **KuaiRec dataset**, a large-scale, fully-observed dataset collected from the Kuaishou short-video platform.

It contains:

- **User interactions** (views, likes, etc.)
- **Video metadata** (video ID, tags, etc.)
- **Timestamps**

More info: [KuaiRec Paper](https://arxiv.org/abs/2202.10842)

**Download dataset**

1. <ins>First option : Downloading Dataset via wget<ins>

In [2]:
%%bash
if [ ! -d "./data_final_project" ]; then
  wget --no-check-certificate 'https://drive.usercontent.google.com/download?id=1qe5hOSBxzIuxBb1G_Ih5X-O65QElollE&export=download&confirm=t&uuid=b2002093-cc6e-4bd5-be47-9603f0b33470' -O KuaiRec.zip
  unzip KuaiRec.zip -d ../data_final_project
else
  echo "Directory './data_final_project' already exists. Skipping download."
fi

Directory './data_final_project' already exists. Skipping download.


2. <ins>Second option : Downloading dataset via Google Drive<ins>

If the data is not downloaded by the wget because of a Connection Refused you might download it via this [link](https://drive.google.com/file/d/1qe5hOSBxzIuxBb1G_Ih5X-O65QElollE/view) and place it at the project root.

In [7]:
""" Uncomment if you need to (and if the first option did not work correctly)
%%bash
unzip KuaiRec.zip -d data_final_project
mv "data_final_project/KuaiRec 2.0/" data_final_project/KuaiRec
"""

' Uncomment if you need to (and if the first option did not work correctly)\n%%bash\nunzip KuaiRec.zip -d data_final_project\nmv "data_final_project/KuaiRec 2.0/" data_final_project/KuaiRec\n'

From this dataset we obtain the following files :

```bash
KuaiRec
  ├── data
  │   ├── big_matrix.csv          
  │   ├── small_matrix.csv
  │   ├── social_network.csv
  │   ├── user_features.csv
  │   ├── item_daily_features.csv
  │   └── item_categories.csv
  │   └── kuairec_caption_category.csv
```

- `interactions_train.csv`: historical user-item interactions for training.
- `interactions_test.csv`: user-item pairs to score during testing.
- `sample_submission.csv`: a template showing the expected output format.
- `video_metadata.csv`: metadata including tags or content-related features.

![image](img/KuaiRec.png)

## **1️⃣ Dataset Preprocessing**
📝 Associated Tasks :
- Load and inspect the dataset.
- Handle missing or inconsistent data.
- Merge metadata for content-based models if necessary.


💡 For more details about the different analyses and pre-processing decisions made on the available datasets, [go to the EDA notebook](./EDA/interactions_EDA.ipynb).

### Load Datasets

In [3]:
interactions = pd.read_csv("./data_final_project/KuaiRec/data/big_matrix.csv")
small_interactions = pd.read_csv("./data_final_project/KuaiRec/data/small_matrix.csv")
item_features = pd.read_csv("./data_final_project/KuaiRec/data/item_daily_features.csv")

def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()  
    return df  

def clean_df_timestamp(df):
    df = clean_df(df)
    df = df[df["timestamp"] >= 0]
    return df

item_features = clean_df(item_features)
item_features = item_features.drop_duplicates(subset='video_id')
train_df = clean_df_timestamp(interactions)
test_df = clean_df_timestamp(small_interactions)

item_features['upload_dt'] = pd.to_datetime(item_features['upload_dt'])
item_features['date'] = pd.to_datetime(item_features['date'], format='%Y%m%d')


In [4]:
to_drop = ['date', 'play_duration', 'video_duration', 'time', 'timestamp']

train_df.drop(columns=to_drop, inplace=True, errors='ignore')
test_df.drop(columns=to_drop, inplace=True, errors='ignore')

In [5]:
correlation = item_features[[
       'video_duration', 'video_width',
       'video_height', 'music_id',
       'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
       'play_duration', 'complete_play_cnt', 'complete_play_user_num',
       'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
       'long_time_play_user_num', 'short_time_play_cnt',
       'short_time_play_user_num', 'play_progress', 'comment_stay_duration',
       'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       'cancel_collect_user_num']].corr()

upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]
# These columns are dropped as we don't need them anymore 
# (merged in previous cell or just not needed for collaborative-filtering)
item_features.drop(columns=to_drop, inplace=True, errors='ignore')


## **2️⃣ Feature Engineering**
📝 Associated Tasks :
- Create meaningful features from interaction and metadata (e.g., content tags, user activity history).
- Build user-item interaction matrix.
- Optionally extract time-based or popularity-based features.

From what we saw on the different analysis, we can conclude that :


In [6]:
item_features['video_age'] = (item_features['date'] - item_features['upload_dt']).dt.days
item_features['is_short_video'] = (item_features['video_duration'].fillna(0) <= 30).astype(int)

We remove those columns as we don't need them in the future.

In [7]:
to_drop = ['date', 'upload_dt', 'video_duration', 'music_id',
                'video_tag_name', 'play_progress', 'video_tag_id',
                'time', 'play_duration'
                ]
item_features.drop(columns=to_drop, inplace=True, errors='ignore')

In [8]:
train_df = pd.merge(train_df, item_features, on='video_id', how='left')
test_df = pd.merge(test_df, item_features, on='video_id', how='left')

As the ALS need some form of rating, we will use what we will call an engagement score. This is what the model will try to predict.

Based on our observations in the [EDA notebook](./EDA/interactions_EDA.ipynb), we can add some features such as :
- watch_ratio 
- video_type => if it is an "AD" then user are less likely to watch the video
- video_height and video_width => `1280x720` is the preferred format
- video_type => `ShortImports` are the preferred format
- visible_status => `public` video are more likely to be seen

In [ ]:
def build_engagement_score(df):
    df["engagement_score"] = 0
    
    # Watch Ratio
    if 'watch_ratio' in df.columns:
        df["engagement_score"] += df['watch_ratio'].fillna(0) * 10
    
    # is_short_video
    df["engagement_score"] += df['is_short_video'].fillna(0) * 3
    
    # Video age
    if 'video_age' in df.columns:
        max_age = 365
        normalized_age = np.minimum(df['video_age'].fillna(max_age), max_age) / max_age
        # Newer videos get up to 2 points bonus
        df["engagement_score"] += (1 - normalized_age) * 2
    
    
    if 'video_type' in df.columns:
        df["engagement_score"] += np.where(
            df['video_type'] == 'AD',
            -3,  # penalty for ads
            2    # bonus for regular content
        )
    
    if 'visible_status' in df.columns:
        df["engagement_score"] += np.where(
            df['visible_status'] == 'public',
            2,  
            -1
        )
    
    if 'upload_type' in df.columns:
        upload_type_weights = {
            'ShortImport': 3,     # Short imported videos tend to be high quality
            'StartCamera': 2.5,   # Original camera content
            'Knowle': 2,          # Knowledge content
            'Web': 1.5,           # Web content
            'LongImport': 1,      # Long imported videos
            'UNKNOWN': 0,
            'LongCamera': 0.5,
            'PictureSet': 0.5,
            'LongPicture': 0.5,
            'ACurlVideo': 0.5,
            'followShot': 0.5,
            'ShareFromOtherApp': 0.5,
            'SameFrame': 0,
            'PictureCopy': 0,
            'FlashPhoto': 0,
            'PhotoCopy': 0,
            'LocalCollection': 0,
            'LocalInteraction': 0
        }
        df["engagement_score"] += df['upload_type'].map(upload_type_weights).fillna(0)
    
    engagement_columns = {
        'like_cnt': 0.5,
        'comment_cnt': 0.7,
        'share_cnt': 0.8,
        'collect_cnt': 0.6,
        'follow_cnt': 0.9,
        'complete_play_cnt': 0.7,
        'valid_play_cnt': 0.5,
        'reply_comment_cnt': 0.6,
        'comment_like_cnt': 0.4
    }
    
    for col, weight in engagement_columns.items():
        if col in df.columns:
            df["engagement_score"] += np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)


    penalty_columns = {
        'cancel_like_cnt': 0.4,
        'cancel_follow_cnt': 0.5,
        'report_cnt': 0.7
    }
    for col, weight in penalty_columns.items():
        if col in df.columns:
            df["engagement_score"] -= np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)

    if 'video_width' in df.columns:
        df["engagement_score"] += np.where(df['video_width'] >= 720, 0.5, 0)

    if 'video_height' in df.columns:
        df["engagement_score"] += np.where(df['video_height'] >= 1280, 0.5, 0)
    
    # Here we normalize the score    
    min_score = df["engagement_score"].min()
    max_score = df["engagement_score"].max()

    df["engagement_score"] = (df["engagement_score"] - min_score) / (max_score - min_score)
    
    return df

test_df = build_engagement_score(test_df)
train_df = build_engagement_score(train_df)

KeyError: 'is_short_video'

We can drop columns that have build the engagement score,
as it will make the following cells run faster (less data to store) 
and prevent the kernel from crashing

In [11]:
test_df.columns

Index(['user_id', 'video_id', 'watch_ratio', 'author_id', 'video_type',
       'upload_type', 'visible_status', 'video_width', 'video_height',
       'show_cnt', 'comment_stay_duration', 'cancel_like_cnt', 'comment_cnt',
       'reply_comment_cnt', 'comment_like_cnt', 'cancel_follow_cnt',
       'share_cnt', 'report_cnt', 'video_age', 'is_short_video',
       'engagement_score'],
      dtype='object')

## **3️⃣ Model Development**
📝 Associated Tasks :
- Choose a recommendation approach:
    - Collaborative filtering (e.g., ALS, Matrix Factorisation)
    - Content-based filtering
    - Sequence-aware models
    - Hybrid approaches
- Train and validate your model on the training set.

In [14]:
# Get Unique user and videos
user_ids_train = train_df['user_id'].unique()
video_ids_train = train_df['video_id'].unique()

# Compute index for each user and videos
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids_train)}
video_to_index = {video_id: idx for idx, video_id in enumerate(video_ids_train)}

# add the index to the train and test
train_df['user_index'] = train_df['user_id'].map(user_to_index)
train_df['video_index'] = train_df['video_id'].map(video_to_index)

test_df['user_index'] = test_df['user_id'].map(user_to_index)
test_df['video_index'] = test_df['video_id'].map(video_to_index)

row = train_df['user_index'].values
col = train_df['video_index'].values

data = train_df['engagement_score'].values

n_users = train_df['user_index'].max() + 1
n_items = train_df['video_index'].max() + 1
    
user_item_matrix = csr_matrix((data, (row, col)), shape=(n_users, n_items))

In [15]:
R = (user_item_matrix != 0).astype(float)

def normalize_ratings(Y, R):
    Ymean = np.zeros(Y.shape[0])
    for i in range(Y.shape[0]):
        if np.sum(R[i, :]) > 0:  # Check if user has any ratings
            Ymean[i] = np.sum(Y[i, :] * R[i, :]) / np.sum(R[i, :])
    
    Ynorm = np.zeros_like(Y)
    for i in range(Y.shape[0]):
        Ynorm[i, :] = (Y[i, :] - Ymean[i]) * R[i, :]
        
    return Ynorm, Ymean

Y_dense = user_item_matrix.toarray()
R_dense = R.toarray()

Ynorm, Ymean = normalize_ratings(Y_dense, R_dense)
user_item_matrix = csr_matrix(Ynorm)

In [16]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=15,
    regularization=0.2,
    iterations=15,
    use_gpu=False,
    alpha=10
)

model.fit(user_item_matrix.T) 

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.05218100547790527 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

## **4️⃣ Recommendation Algorithm**
📝 Associated Tasks :
- Predict which videos are likely to be enjoyed by each user in the test set.
- Generate a top-N ranked list of recommendations for each user.

In [17]:
top_n=100
def get_top_n_recommendations(model, user_item_matrix, user_ids, n=10, seen=True):
    recommendations = {}
    
    for user_id in user_ids:
        # Items the user has already interacted with (adjusted for training)
        already_interacted = set(user_item_matrix[user_id].indices) if seen else set()
        
        # U * V^T
        scores = model.user_factors[user_id].dot(model.item_factors.T) + Ymean[user_id]
       
        item_scores = [(item_id, scores[item_id])
                       for item_id in range(len(scores))
                       if item_id not in already_interacted]
        # Sort and select top-N items
        item_scores.sort(key=lambda x: x[1], reverse=True)
        top_items = [item[0] for item in item_scores[:n]]
        
        recommendations[user_id] = top_items
    print(item_scores[::-1])
    return recommendations

train_users = train_df['user_index'].unique()
test_users = test_df['user_index'].unique()

# For training eval, exclude seen items to simulate a true recommendation
train_recommendations = get_top_n_recommendations(
    model, user_item_matrix, train_users, n=top_n, seen=False  # Exclude seen items in training
)

# For test eval, include only unseen items (as per real-world recommendation)
test_recommendations = get_top_n_recommendations(
    model, user_item_matrix, test_users, n=top_n, seen=True  # Exclude seen items in test (real-world)
)

print(test_recommendations)

[(3586, 0.0046565044), (6792, 0.0052060112), (353, 0.0052194484), (3431, 0.0052198884), (5991, 0.005235257), (2234, 0.0052555464), (6863, 0.0052764816), (4029, 0.0052909004), (5384, 0.0053108186), (4035, 0.0053136917), (6107, 0.0053182417), (4947, 0.005327086), (2583, 0.005330422), (1569, 0.0053424663), (3816, 0.0053470368), (4589, 0.0053571793), (5249, 0.0053590485), (6738, 0.0053605735), (3771, 0.0053645037), (6570, 0.0053656166), (4825, 0.0053658145), (4436, 0.0053663435), (6800, 0.005373602), (5500, 0.005374131), (2353, 0.005381146), (816, 0.0053829197), (799, 0.005384802), (1090, 0.005385464), (5476, 0.0053900164), (2168, 0.0053912555), (6703, 0.0053922557), (3162, 0.005394959), (3752, 0.0054011215), (3389, 0.005401261), (2213, 0.005401794), (5684, 0.0054059555), (383, 0.005407747), (2546, 0.005408423), (4927, 0.0054091937), (5816, 0.005409847), (3180, 0.0054116137), (5412, 0.0054133274), (600, 0.0054151295), (4159, 0.005415758), (510, 0.005416421), (5894, 0.00542199), (4485, 0.00

## **5️⃣ Evaluation**
📝 Associated Tasks :
- Choose suitable metrics (e.g., Precision@K, Recall@K, MAP, NDCG).
- Evaluate performance and provide interpretations.

In [18]:
import numpy as np
from scipy.sparse import csr_matrix

def evaluate_recommendations_with_additional_metrics(recommendations, test_df, top_n=10, k=10):
    # Map of actual items per user
    user_actual_items = test_df.groupby('user_index')['video_index'].apply(set).to_dict()
    
    precision_at_n = []
    recall_at_n = []

    # Hit Rate, MRR, nDCG calculations
    hits = 0
    mrr = 0.0
    total_ndcg = 0.0
    count = 0
    
    for user_id, recommended_items in recommendations.items():
        if user_id in user_actual_items:
            actual_items = user_actual_items[user_id]
            recs_at_n = recommended_items[:top_n]

            # Precision and Recall at N
            num_relevant = len(set(recs_at_n) & actual_items)
            precision = num_relevant / len(recs_at_n) if recs_at_n else 0
            recall = num_relevant / len(actual_items) if actual_items else 0

            precision_at_n.append(precision)
            recall_at_n.append(recall)
            
            # Hit Rate
            gt = user_actual_items.get(user_id, set())
            if any(item in gt for item in recs_at_n[:k]):
                hits += 1

            # MRR
            for rank, item in enumerate(recs_at_n[:k], start=1):
                if item in gt:
                    mrr += 1.0 / rank
                    break

            # nDCG
            def dcg(recs, gt, k):
                return sum((1 / np.log2(i + 2)) if rec in gt else 0 for i, rec in enumerate(recs[:k]))

            def idcg(gt, k):
                n_relevant = min(len(gt), k)
                return sum(1 / np.log2(i + 2) for i in range(n_relevant))

            idcg_val = idcg(gt, k)
            if idcg_val > 0:
                total_ndcg += dcg(recs_at_n, gt, k) / idcg_val
                count += 1

    avg_precision = np.mean(precision_at_n) if precision_at_n else 0
    avg_recall = np.mean(recall_at_n) if recall_at_n else 0
    hit_rate = hits / len(user_actual_items) if user_actual_items else 0
    avg_mrr = mrr / len(user_actual_items) if user_actual_items else 0
    avg_ndcg = total_ndcg / count if count > 0 else 0

    return avg_precision, avg_recall, hit_rate, avg_mrr, avg_ndcg


# Create test user-item matrix for ground truth
test_row = test_df['user_index'].values
test_col = test_df['video_index'].values
test_data = np.ones(len(test_df))  # binary implicit feedback

test_user_item_matrix = csr_matrix((test_data, (test_row, test_col)), shape=(n_users, n_items))

def sparse_matrix_to_dict(matrix):
    user_item_dict = {}
    for user_id in range(matrix.shape[0]):
        items = matrix[user_id].indices
        if len(items) > 0:
            user_item_dict[user_id] = set(items)
    return user_item_dict

# Get ground truth from test matrix
ground_truth = sparse_matrix_to_dict(test_user_item_matrix)

# Evaluate on training and testing set
print("Evaluating on training set...")
train_precision, train_recall, train_hit_rate, train_mrr, train_ndcg = evaluate_recommendations_with_additional_metrics(train_recommendations, train_df, top_n=top_n, k=10)
print(f"Training - Precision@{top_n}: {train_precision:.4f}, Recall@{top_n}: {train_recall:.4f}, Hit Rate@{10}: {train_hit_rate:.4f}, MRR@{10}: {train_mrr:.4f}, nDCG@{10}: {train_ndcg:.4f}")

print("Evaluating on test set...")
test_precision, test_recall, test_hit_rate, test_mrr, test_ndcg = evaluate_recommendations_with_additional_metrics(test_recommendations, test_df, top_n=top_n, k=10)
print(f"Testing - Precision@{top_n}: {test_precision:.4f}, Recall@{top_n}: {test_recall:.4f}, Hit Rate@{10}: {test_hit_rate:.4f}, MRR@{10}: {test_mrr:.4f}, nDCG@{10}: {test_ndcg:.4f}")


Evaluating on training set...
Training - Precision@100: 0.1831, Recall@100: 0.0129, Hit Rate@10: 0.7740, MRR@10: 0.3455, nDCG@10: 0.1696
Evaluating on test set...
Testing - Precision@100: 0.4291, Recall@100: 0.0135, Hit Rate@10: 0.9965, MRR@10: 0.6125, nDCG@10: 0.3769
